# Task 1 — Exploratory Data Analysis
Financial news dataset: descriptive stats, topic modeling, time-series publication analysis, publisher analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# NLP
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords

plt.style.use('seaborn-v0_8-whitegrid')
STOP_WORDS = set(stopwords.words('english'))

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/raw/raw_analyst_ratings.csv')
print(df.shape)
df.head()

In [ ]:
# Parse dates — handle mixed timezone formats
df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce')
df.dropna(subset=['date'], inplace=True)
df['date_only'] = df['date'].dt.date
df['hour'] = df['date'].dt.hour
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['dayofweek'] = df['date'].dt.day_name()

print(f"Date range: {df['date'].min()} → {df['date'].max()}")
print(f"Missing values:\n{df.isnull().sum()}")

## 2. Descriptive Statistics

In [ ]:
df['headline_len'] = df['headline'].str.len()
df['word_count'] = df['headline'].str.split().str.len()

print("=== Headline Character Count ===")
print(df['headline_len'].describe().round(2))
print("\n=== Word Count ===")
print(df['word_count'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['headline_len'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Headline Character Length Distribution')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Count')

axes[1].hist(df['word_count'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Headline Word Count Distribution')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/raw/fig1_headline_distributions.png', dpi=150)
plt.show()

## 3. Publisher Analysis

In [ ]:
# Extract domain if publisher looks like an email
def extract_domain(pub):
    if isinstance(pub, str) and '@' in pub:
        return pub.split('@')[-1].lower()
    return pub

df['publisher_clean'] = df['publisher'].apply(extract_domain)

top_publishers = df['publisher_clean'].value_counts().head(15)
print(top_publishers)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
top_publishers.plot(kind='barh', ax=ax, color='teal')
ax.set_title('Top 15 Most Active Publishers')
ax.set_xlabel('Article Count')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../data/raw/fig2_top_publishers.png', dpi=150)
plt.show()

## 4. Time Series Analysis of News Volume

In [ ]:
daily_counts = df.groupby('date_only').size().reset_index(name='count')
daily_counts['date_only'] = pd.to_datetime(daily_counts['date_only'])

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_counts['date_only'], daily_counts['count'], linewidth=0.8, color='navy', alpha=0.7)
ax.fill_between(daily_counts['date_only'], daily_counts['count'], alpha=0.2, color='navy')
ax.set_title('Daily Article Publication Volume Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Articles Published')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/raw/fig3_daily_volume.png', dpi=150)
plt.show()

In [ ]:
# Publishing hour distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Articles by Hour of Day (UTC)')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Count')

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df['dayofweek'].value_counts().reindex(day_order).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Articles by Day of Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/raw/fig4_publishing_times.png', dpi=150)
plt.show()

## 5. Text Analysis — Keywords & Topic Modeling

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [w for w in text.split() if w not in STOP_WORDS and len(w) > 2]
    return ' '.join(tokens)

df['clean_headline'] = df['headline'].apply(clean_text)

# Top TF-IDF terms
tfidf = TfidfVectorizer(max_features=30, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['clean_headline'])
tfidf_scores = tfidf_matrix.mean(axis=0).A1
tfidf_terms = pd.Series(tfidf_scores, index=tfidf.get_feature_names_out()).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
tfidf_terms.head(20).plot(kind='barh', ax=ax, color='mediumseagreen')
ax.set_title('Top 20 TF-IDF Terms in Headlines')
ax.set_xlabel('Mean TF-IDF Score')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../data/raw/fig5_tfidf_terms.png', dpi=150)
plt.show()

In [ ]:
# LDA Topic Modeling
cv = CountVectorizer(max_features=500, ngram_range=(1, 2))
cv_matrix = cv.fit_transform(df['clean_headline'])

lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=10)
lda.fit(cv_matrix)

feature_names = cv.get_feature_names_out()
print("=== LDA Topics ===")
for i, topic in enumerate(lda.components_):
    top_words = [feature_names[j] for j in topic.argsort()[-10:][::-1]]
    print(f"Topic {i+1}: {', '.join(top_words)}")

## 6. Stock Coverage Analysis

In [ ]:
top_stocks = df['stock'].value_counts().head(20)
print(f"Unique stocks covered: {df['stock'].nunique()}")

fig, ax = plt.subplots(figsize=(10, 5))
top_stocks.plot(kind='bar', ax=ax, color='slateblue')
ax.set_title('Top 20 Most Covered Stocks')
ax.set_xlabel('Stock Ticker')
ax.set_ylabel('Article Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/raw/fig6_top_stocks.png', dpi=150)
plt.show()